# Setup & Validation

This notebook verifies the `collatz` library against the worked examples in the
accompanying papers. Each assertion below corresponds to a known result; if every
cell runs without error the library is consistent with the published tables and
formulas.

In [1]:
import sys, pathlib
# Ensure the repo root is on the path so `import collatz` works
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from collatz.core import collatz_step, orbit, total_stopping_time, stopping_time, stopping_destination, stopping_orbit
from collatz.dropping import (
    dropping_time, dropping_destination, dropping_orbit, dropping_set,
    orbital_oddity, dropping_modulus, dropping_index, dropping_genus, classify_range as dropping_classify_range,
)
from collatz.stopping import (
    stopping_time as st_stopping_time, stopping_destination as st_stopping_dest,
    stopping_orbit as st_stopping_orbit, stopping_point, stopping_class,
    stopping_modulus_for_class, stopping_page, stopping_offset, stopping_signature,
    stopping_line_slope, classify_range as stopping_classify_range,
)
from collatz.geometry import (
    orbital_triple, incircle_params, circumcircle_params, stopping_circle,
    complex_multiplier, complex_multiplier_exact, proportional_power_ratio,
)
from collatz.factorization import (
    prime_factorization, is_prime, prime_class_character,
    class_character_signature, compare_composite_vs_factors,
    factor_classification_table, shared_class_primes, class_distribution_by_prime,
)

print("All collatz modules imported successfully.")

All collatz modules imported successfully.


## Core Functions

In [2]:
# Verify orbit(27) starts with [27, 82, 41, ...]
o27 = orbit(27)
assert o27[:3] == [27, 82, 41], f"orbit(27) starts with {o27[:3]}"
print(f"orbit(27) starts with {o27[:3]}  (length {len(o27)})")

# Verify total_stopping_time(27) == 111
tst27 = total_stopping_time(27)
assert tst27 == 111, f"total_stopping_time(27) = {tst27}"
print(f"total_stopping_time(27) = {tst27}")

print("\nCore function checks passed.")

orbit(27) starts with [27, 82, 41]  (length 112)
total_stopping_time(27) = 111

Core function checks passed.


## Dropping (Paper 1)

In [3]:
# dropping_genus(3) == (6, 0, 0)
dg3 = dropping_genus(3)
assert dg3 == (6, 0, 0), f"dropping_genus(3) = {dg3}"
print(f"dropping_genus(3) = {dg3}")

# dropping_orbit(3) == [3, 10, 5, 16, 8, 4]
do3 = dropping_orbit(3)
assert do3 == [3, 10, 5, 16, 8, 4], f"dropping_orbit(3) = {do3}"
print(f"dropping_orbit(3) = {do3}")

# dropping_orbit(5) == [5, 16, 8]
do5 = dropping_orbit(5)
assert do5 == [5, 16, 8], f"dropping_orbit(5) = {do5}"
print(f"dropping_orbit(5) = {do5}")

# orbital_oddity(3) == 2, orbital_oddity(5) == 1
oo3 = orbital_oddity(3)
oo5 = orbital_oddity(5)
assert oo3 == 2, f"orbital_oddity(3) = {oo3}"
assert oo5 == 1, f"orbital_oddity(5) = {oo5}"
print(f"orbital_oddity(3) = {oo3}")
print(f"orbital_oddity(5) = {oo5}")

# Table 5, Paper 1: mapping of dropping set -> orbital oddity
# {dropping_set: expected_orbital_oddity}
table5 = {1: 0, 3: 1, 6: 2, 8: 3, 11: 4, 13: 5, 16: 6}
print("\nVerifying Table 5 (dropping set -> orbital oddity):")
for ds, expected_oo in table5.items():
    # Find one representative member of each dropping set
    rep = next(n for n in range(2, 500) if dropping_set(n) == ds)
    actual_oo = orbital_oddity(rep)
    assert actual_oo == expected_oo, (
        f"Dropping Set {ds}: expected oo={expected_oo}, got {actual_oo} (rep n={rep})"
    )
    print(f"  Dropping Set {ds:>2}  ->  orbital oddity {actual_oo}  (representative n={rep})")

print("\nDropping checks passed.")

dropping_genus(3) = (6, 0, 0)
dropping_orbit(3) = [3, 10, 5, 16, 8, 4]
dropping_orbit(5) = [5, 16, 8]
orbital_oddity(3) = 2
orbital_oddity(5) = 1

Verifying Table 5 (dropping set -> orbital oddity):
  Dropping Set  1  ->  orbital oddity 0  (representative n=2)
  Dropping Set  3  ->  orbital oddity 1  (representative n=5)
  Dropping Set  6  ->  orbital oddity 2  (representative n=3)
  Dropping Set  8  ->  orbital oddity 3  (representative n=11)
  Dropping Set 11  ->  orbital oddity 4  (representative n=7)
  Dropping Set 13  ->  orbital oddity 5  (representative n=39)
  Dropping Set 16  ->  orbital oddity 6  (representative n=287)

Dropping checks passed.


## Stopping (Paper 2)

In [4]:
# stopping_point(19) == (-8, 11)
sp19 = stopping_point(19)
assert sp19 == (-8, 11), f"stopping_point(19) = {sp19}"
print(f"stopping_point(19) = {sp19}")

# Table 1, Paper 2: stopping points for the first 16 odd numbers >= 3
print("\nTable 1 -- stopping points for first 16 odd numbers >= 3:")
odd_nums = [n for n in range(3, 100) if n % 2 == 1][:16]
print(f"{'n':>4}  {'stop_pt':>12}  {'class':>5}")
for n in odd_nums:
    sp = stopping_point(n)
    sc = stopping_class(n)
    print(f"{n:>4}  {str(sp):>12}  {sc:>5}")

# Stopping signatures for class 8
print("\nStopping signatures for Stopping Class 8:")
class8_expected = {
    11:  (8, 1, 0),
    23:  (8, 1, 1),
    43:  (8, 2, 0),
}
for n, expected_sig in class8_expected.items():
    sig = stopping_signature(n)
    assert sig == expected_sig, f"stopping_signature({n}) = {sig}, expected {expected_sig}"
    print(f"  n={n:>3}  ->  signature {sig}")

# Stopping line slopes for classes 3, 6, 8
print("\nStopping line slopes:")
for k in [3, 6, 8]:
    slope = stopping_line_slope(k)
    print(f"  Class {k}: slope = {slope} = {float(slope):.6f}")

print("\nStopping checks passed.")

stopping_point(19) = (-8, 11)

Table 1 -- stopping points for first 16 odd numbers >= 3:
   n       stop_pt  class
   3       (-1, 2)      6
   5       (-1, 4)      3
   7       (-2, 5)     11
   9       (-2, 7)      3
  11      (-1, 10)      8
  13      (-3, 10)      3
  15      (-5, 10)     11
  17      (-4, 13)      3
  19      (-8, 11)      6
  21      (-5, 16)      3
  23      (-3, 20)      8
  25      (-6, 19)      3
  27      (-4, 23)     96
  29      (-7, 22)      3
  31      (-8, 23)     91
  33      (-8, 25)      3

Stopping signatures for Stopping Class 8:
  n= 11  ->  signature (8, 1, 0)
  n= 23  ->  signature (8, 1, 1)
  n= 43  ->  signature (8, 2, 0)

Stopping line slopes:
  Class 3: slope = -3 = -3.000000
  Class 6: slope = -9/7 = -1.285714
  Class 8: slope = -5 = -5.000000

Stopping checks passed.


## Geometry

In [5]:
# orbital_triple(3) == (5, 12, 13)
ot3 = orbital_triple(3)
assert ot3 == (5, 12, 13), f"orbital_triple(3) = {ot3}"
print(f"orbital_triple(3) = {ot3}")

# Verify it is a Pythagorean triple
a, b, c = ot3
assert a**2 + b**2 == c**2, f"{a}^2 + {b}^2 != {c}^2"
print(f"  {a}^2 + {b}^2 = {a**2 + b**2} = {c}^2  (Pythagorean triple confirmed)")

# complex_multiplier(3)
z3 = complex_multiplier(3)
print(f"\ncomplex_multiplier(3)  = {z3}")
re3, im3 = complex_multiplier_exact(3)
print(f"  exact: ({re3}, {im3})")

# complex_multiplier(27)
z27 = complex_multiplier(27)
print(f"\ncomplex_multiplier(27) = {z27}")
re27, im27 = complex_multiplier_exact(27)
print(f"  exact: ({re27}, {im27})")

print("\nGeometry checks passed.")

orbital_triple(3) = (5, 12, 13)
  5^2 + 12^2 = 169 = 13^2  (Pythagorean triple confirmed)

complex_multiplier(3)  = (0.5+0.16666666666666666j)
  exact: (1/2, 1/6)

complex_multiplier(27) = (0.5+0.35185185185185186j)
  exact: (1/2, 19/54)

Geometry checks passed.


## All Checks Passed

Every assertion succeeded. The library is consistent with the paper examples.